# 📊 Supermarket Shelves: A Beginner's Guide to Parquet, ORC, and Arrow

Welcome! In our previous lessons, we learned how to "pack" and "ship" individual records. Now, we're looking at how to organize **massive amounts of data** for analysis.

### The Big Idea: Row-Based vs. Columnar Storage
Imagine you are in a massive supermarket:

- **Row-Based (CSV, JSON, Avro)**: This is like having a billion individual shopping receipts. If your boss asks, "What was the total spent on milk last year?", you have to pick up every single receipt, find the milk entry, and add it up. It takes forever because you're reading information about bread, eggs, and soap that you don't need.
- **Columnar (Parquet, ORC)**: This is like having the supermarket organized into **aisles**. If you need the total for milk, you go straight to the "Milk Prices" column. You completely ignore the rest of the store. This makes queries **lightning fast** and saves a lot of storage space.

## 1. What are Parquet, ORC, and Arrow?

These are "Columnar" formats:
1.  **Apache Parquet**: The industry standard for data at rest (files in a lake). It's great for high-performance analytics.
2.  **Apache ORC**: Similar to Parquet, often used in Hadoop/Hive environments.
3.  **Apache Arrow**: This is about data **in memory**. It allows different programs (like Python and Spark) to share the same data instantly without "packing and unpacking" it.

### Reading (Decoding) and Writing (Encoding) a Parquet File

Let's look at how to decode and encode a Parquet file with mock customers data. We use the `pyarrow` library, which is the "forklift" that moves these columnar tables around.

In [5]:
import pyarrow as pa
import pyarrow.parquet as pq
from pprint import pprint

In [6]:
table = pq.read_table('../data/userdata1.parquet')

In [9]:
table

pyarrow.Table
registration_dttm: timestamp[ns]
id: int32
first_name: string
last_name: string
email: string
gender: string
ip_address: string
cc: string
country: string
birthdate: string
salary: double
title: string
comments: string
----
registration_dttm: [[2016-02-03 07:55:29.000000000,2016-02-03 17:04:03.000000000,2016-02-03 01:09:31.000000000,2016-02-03 00:36:21.000000000,2016-02-03 05:05:31.000000000,...,2016-02-03 10:30:59.000000000,2016-02-03 17:16:53.000000000,2016-02-03 05:02:20.000000000,2016-02-03 02:41:32.000000000,2016-02-03 09:52:18.000000000]]
id: [[1,2,3,4,5,...,996,997,998,999,1000]]
first_name: [["Amanda","Albert","Evelyn","Denise","Carlos",...,"Dennis","Gloria","Nancy","Annie","Julie"]]
last_name: [["Jordan","Freeman","Morgan","Riley","Burns",...,"Harris","Hamilton","Morris","Daniels","Meyer"]]
email: [["ajordan0@com.com","afreeman1@is.gd","emorgan2@altervista.org","driley3@gmpg.org","cburns4@miitbeian.gov.cn",...,"dharrisrn@eepurl.com","ghamiltonro@rambler.ru","nmor

In [14]:
table['salary']

[
  [
    49756.53,
    150280.17,
    144972.51,
    90263.05,
    null,
    ...
    263399.54,
    83183.54,
    null,
    18433.85,
    222561.13
  ]
]

### The Schema: The Store Directory
The schema tells us what columns (aisles) are available in our data.

In [10]:
table.schema

registration_dttm: timestamp[ns]
id: int32
first_name: string
last_name: string
email: string
gender: string
ip_address: string
cc: string
country: string
birthdate: string
salary: double
title: string
comments: string

### Metadata: The Labels on the Aisle
Parquet files store "Metadata." This is information *about* the data (like the minimum and maximum value in a column) without actually reading the data itself. This helps tools skip huge chunks of files that aren't relevant to your search.

In [15]:
metadata = pq.read_metadata('../data/userdata1.parquet')

metadata

  created_by: parquet-mr version 1.8.1 (build 4aba4dae7bb0d4edbcf7923ae1339f28fd3f7fcf)
  num_columns: 13
  num_rows: 1000
  num_row_groups: 1
  format_version: 1.0
  serialized_size: 1125

In [16]:
metadata.schema

required group field_id=-1 hive_schema {
  optional int96 field_id=-1 registration_dttm;
  optional int32 field_id=-1 id;
  optional binary field_id=-1 first_name (String);
  optional binary field_id=-1 last_name (String);
  optional binary field_id=-1 email (String);
  optional binary field_id=-1 gender (String);
  optional binary field_id=-1 ip_address (String);
  optional binary field_id=-1 cc (String);
  optional binary field_id=-1 country (String);
  optional binary field_id=-1 birthdate (String);
  optional double field_id=-1 salary;
  optional binary field_id=-1 title (String);
  optional binary field_id=-1 comments (String);
}


In [20]:
metadata.row_group(0).column(12)

  file_offset: 108208
  file_path: 
  physical_type: BYTE_ARRAY
  num_values: 1000
  path_in_schema: comments
  is_stats_set: False
  statistics:
    None
  geo_statistics:
    None
  compression: UNCOMPRESSED
  encodings: ('BIT_PACKED', 'PLAIN_DICTIONARY', 'RLE')
  has_dictionary_page: False
  dictionary_page_offset: None
  data_page_offset: 108208
  total_compressed_size: 4288
  total_uncompressed_size: 4288

Select the first 3 rows of the table:

In [21]:
table.take([0,1,2])

pyarrow.Table
registration_dttm: timestamp[ns]
id: int32
first_name: string
last_name: string
email: string
gender: string
ip_address: string
cc: string
country: string
birthdate: string
salary: double
title: string
comments: string
----
registration_dttm: [[2016-02-03 07:55:29.000000000,2016-02-03 17:04:03.000000000,2016-02-03 01:09:31.000000000]]
id: [[1,2,3]]
first_name: [["Amanda","Albert","Evelyn"]]
last_name: [["Jordan","Freeman","Morgan"]]
email: [["ajordan0@com.com","afreeman1@is.gd","emorgan2@altervista.org"]]
gender: [["Female","Male","Female"]]
ip_address: [["1.197.201.2","218.111.175.34","7.161.136.94"]]
cc: [["6759521864920116","","6767119071901597"]]
country: [["Indonesia","Canada","Russia"]]
birthdate: [["3/8/1971","1/16/1968","2/1/1960"]]
...

### Conversion to Pandas (The Analytics Tool)
In Python, we usually do our heavy data lifting in `pandas`. Converting a Parquet table to a Pandas DataFrame is very common.

In [23]:
df = table.to_pandas()

In [10]:
df

,registration_dttm,id,first_name,last_name,email,gender,ip_address,cc,country,birthdate,salary,title,comments
0,2016-02-03 07:55:29,1,Amanda,Jordan,ajordan0@com.com,Female,1.197.201.2,6759521864920116,Indonesia,3/8/1971,49756.53,Internal Auditor,1E+02
1,2016-02-03 17:04:03,2,Albert,Freeman,afreeman1@is.gd,Male,218.111.175.34,,Canada,1/16/1968,150280.17,Accountant IV,
2,2016-02-03 01:09:31,3,Evelyn,Morgan,emorgan2@altervista.org,Female,7.161.136.94,6767119071901597,Russia,2/1/1960,144972.51,Structural Engineer,
3,2016-02-03 00:36:21,4,Denise,Riley,driley3@gmpg.org,Female,140.35.109.83,3576031598965625,China,4/8/1997,90263.05,Senior Cost Accountant,
4,2016-02-03 05:05:31,5,Carlos,Burns,cburns4@miitbeian.gov.cn,,169.113.235.40,5602256255204850,South Africa,,NaN,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2016-02-03 10:30:59,996,Dennis,Harris,dharrisrn@eepurl.com,Male,178.180.111.236,374288806662929,Greece,7/8/1965,263399.54,Editor,
996,2016-02-03 17:16:53,997,Gloria,Hamilton,ghamiltonro@rambler.ru,Female,71.50.39.137,,China,4/22/1975,83183.54,VP Product Management,
997,2016-02-03 05:02:20,998,Nancy,Morris,nmorrisrp@ask.com,,6.188.121.221,3553564071014997,Sweden,5/1/1979,NaN,Junior Executive,
998,2016-02-03 02:41:32,999,Annie,Daniels,adanielsrq@squidoo.com,Female,97.221.132.35,30424803513734,China,10/9/1991,18433.85,Editor,​


You can convert the DataFrame back to a Table (note we're using the method from `pa` which is pyarrow):

In [24]:
new_table = pa.Table.from_pandas(df)

new_table

pyarrow.Table
registration_dttm: timestamp[ns]
id: int32
first_name: string
last_name: string
email: string
gender: string
ip_address: string
cc: string
country: string
birthdate: string
salary: double
title: string
comments: string
----
registration_dttm: [[2016-02-03 07:55:29.000000000,2016-02-03 17:04:03.000000000,2016-02-03 01:09:31.000000000,2016-02-03 00:36:21.000000000,2016-02-03 05:05:31.000000000,...,2016-02-03 10:30:59.000000000,2016-02-03 17:16:53.000000000,2016-02-03 05:02:20.000000000,2016-02-03 02:41:32.000000000,2016-02-03 09:52:18.000000000]]
id: [[1,2,3,4,5,...,996,997,998,999,1000]]
first_name: [["Amanda","Albert","Evelyn","Denise","Carlos",...,"Dennis","Gloria","Nancy","Annie","Julie"]]
last_name: [["Jordan","Freeman","Morgan","Riley","Burns",...,"Harris","Hamilton","Morris","Daniels","Meyer"]]
email: [["ajordan0@com.com","afreeman1@is.gd","emorgan2@altervista.org","driley3@gmpg.org","cburns4@miitbeian.gov.cn",...,"dharrisrn@eepurl.com","ghamiltonro@rambler.ru","nmor

You can write the table back to a Parquet file:

In [25]:
pq.write_table(new_table, "../data/userdata2.parquet")

## 🎓 Beginner Exercises

Try these to see if you've mastered the "Supermarket Layout":

1.  **How many males and females are there?** (Hint: Use `df['gender'].value_counts()`)
2.  **What is the average salary for customers from China?**
3.  **Create a new column `full_name`** which combines `first_name` and `last_name` with a space in between in the dataframe. Then convert it back to a new Table and write it to a Parquet file.

### Reading (Decoding) and Writing (Encoding) an ORC File

Let's look at how to decode and encode an ORC file with mock data. ORC is very similar to Parquet.

In [26]:
import pyarrow as pa
from pyarrow import orc

In [27]:
table2 = orc.read_table('../data/userdata1.1.orc')

In [28]:
table2

pyarrow.Table
registration_dttm: timestamp[ns]
id: double
first_name: string
last_name: string
email: string
gender: string
ip_address: string
cc: string
country: string
birthdate: string
salary: double
title: string
comments: string
----
registration_dttm: [[2016-02-03 13:36:39.000000000,2016-02-03 00:22:28.000000000,2016-02-03 18:29:04.000000000,2016-02-03 13:42:19.000000000,2016-02-03 00:15:29.000000000,...,2016-02-03 13:36:49.000000000,2016-02-03 04:39:01.000000000,2016-02-03 00:33:54.000000000,2016-02-03 00:15:08.000000000,2016-02-03 00:53:53.000000000]]
id: [[1,2,3,4,5,...,996,997,998,999,1000]]
first_name: [["Donald","Walter","Michelle","Lori","Howard",...,"Carol","Helen","Stephanie","Marie","Alice"]]
last_name: [["Lewis","Collins","Henderson","Hudson","Miller",...,"Warren","Fields","Sims","Medina","Peterson"]]
email: [["dlewis0@clickbank.net","wcollins1@bloglovin.com","mhenderson2@geocities.jp","lhudson3@dion.ne.jp","hmiller4@fema.gov",...,"cwarrenrn@geocities.jp","hfieldsro@co

In [29]:
df2 = table2.to_pandas()

df2

,registration_dttm,id,first_name,last_name,email,gender,ip_address,cc,country,birthdate,salary,title,comments
0,2016-02-03 13:36:39,1.0,Donald,Lewis,dlewis0@clickbank.net,Male,102.22.124.20,,Indonesia,7/9/1972,140249.37,Senior Financial Analyst,
1,2016-02-03 00:22:28,2.0,Walter,Collins,wcollins1@bloglovin.com,Male,247.28.26.93,3587726269478025,China,,NaN,,
2,2016-02-03 18:29:04,3.0,Michelle,Henderson,mhenderson2@geocities.jp,Female,193.68.146.150,,France,1/15/1964,236219.26,Teacher,
3,2016-02-03 13:42:19,4.0,Lori,Hudson,lhudson3@dion.ne.jp,,34.252.168.48,3568840151595649,Russia,4/22/1988,NaN,Nuclear Power Engineer,
4,2016-02-03 00:15:29,5.0,Howard,Miller,hmiller4@fema.gov,Male,103.193.150.230,3583473261055014,France,11/26/1998,50210.02,Senior Editor,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2016-02-03 13:36:49,996.0,Carol,Warren,cwarrenrn@geocities.jp,Female,71.7.191.213,,China,,185421.82,,""""
996,2016-02-03 04:39:01,997.0,Helen,Fields,hfieldsro@comcast.net,Female,164.190.97.183,,Malaysia,,279671.68,,
997,2016-02-03 00:33:54,998.0,Stephanie,Sims,ssimsrp@newyorker.com,Female,135.66.68.181,3548125808139842,Poland,,112275.78,,
998,2016-02-03 00:15:08,999.0,Marie,Medina,mmedinarq@thetimes.co.uk,Female,223.83.175.211,,Kazakhstan,3/25/1969,53564.76,Speech Pathologist,


You can write the table back to an ORC file:

In [30]:
orc.write_table(table2, "../data/file2.orc")